In [ ]:
import os

path = "/uscmst1b_scratch/lpc1/htcondor/"

print("CONDOR_CONFIG:", os.environ.get("CONDOR_CONFIG"))
print("Directory exists:", os.path.isdir(path))
print("Read access:", os.access(path, os.R_OK))
print("Write access:", os.access(path, os.W_OK))
print("Search access:", os.access(path, os.X_OK))

In [ ]:
import os
import sys
from datetime import datetime

REPO = "/uscms/home/reshmar/nobackup/IDMe_Run3_Collab/CMSSW_13_0_13/src/iDMe"
AT = os.path.join(REPO, "python_analysis/analysisTools")
CONFIGS = os.path.join(REPO, "python_analysis/configs")

os.environ["X509_USER_PROXY"] = os.path.expanduser("~/x509up_proxy.pem")

if AT not in sys.path:
    sys.path.insert(0, AT)

from lpc_dask import make_lpc_client, ship_analysis_modules
from analysisTools import Analyzer
import coffea.util as util

sample_config = os.path.join(
    CONFIGS, "sample_configs/bkg_2024_Merged_Full_Run3.json"
)
slim_config = os.path.join(
    CONFIGS, "sample_configs/bkg_2024_Slim_Run3.json"
)
cuts_config = os.path.join(CONFIGS, "cut_configs/cutconfig.py")
histos_config = os.path.join(CONFIGS, "histo_configs/histo_cutflow.py")

outdir = os.path.join(REPO, "python_analysis/coffea/SlimPlusMerged2026")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
outfile = os.path.join(outdir, f"Bkg_DASK_{timestamp}.coffea")



analyzer = Analyzer(
    sample_config,
    histos_config,
    cuts_config,
    model_json=None,
    max_samples=-1,
    slimFileList=slim_config,
    processSlim=True,
)

# Analyzer replaces merged-sample normalization metadata with slim metadata.
# full_num_events retains the original merged count.
expected = sum(
    info.get("full_num_events", info["num_events"])
    for info in analyzer.sample_info.values()
)
print("Expected input entries (merged + paired slim):", expected)

os.makedirs(outdir, exist_ok=True)

cluster = None
client = None

try:
    cluster, client = make_lpc_client(n_workers=40)
    print("dashboard:", cluster.dashboard_link)

    # Workers may take time to appear in a busy Condor pool.
    client.wait_for_workers(1, timeout=1800)



    ship_analysis_modules(
        client,
        analysis_tools_dir=AT,
        configs=[cuts_config, histos_config],
    )

    out = analyzer.process(execr="dask", dask_client=client)
    acc, metrics = out

    actual = metrics["entries"]
    print("Processed entries:", actual)
    print("Expected entries:", expected)

    if actual != expected:
        print(
            "WARNING: entry counts differ. Check skipped files and "
            "sample-config bookkeeping before using this output."
        )

    # Exclusive creation prevents overwriting an existing output.
    with open(outfile, "xb") as handle:
        util.save(out, handle)

    print("Saved accumulator and metrics:", outfile)

finally:
    try:
        if client is not None:
            client.close()
    finally:
        if cluster is not None:
            cluster.close()

Expected input entries (merged + paired slim): 4754558041


2026-11-13 15:00:14,184 - tornado.application - ERROR - Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOMainLoop object at 0x7f689b62f010>>, <Task finished name='Task-45' coro=<SpecCluster._correct_state_internal() done, defined at /usr/local/lib/python3.11/site-packages/distributed/deploy/spec.py:352> exception=HTCondorException('Failed to connect to schedd, errmsg=AUTHENTICATE:1003:Failed to authenticate with any method|AUTHENTICATE:1004:Failed to authenticate using FS_REMOTE')>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/tornado/ioloop.py", line 756, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/tornado/ioloop.py", line 780, in _discard_future_result
    future.result()
  File "/usr/local/lib/python3.11/site-packages/distributed/deploy/spec.py", line 396, in _correct_state_internal
    await asyncio.gather(*worke

dashboard: http://localhost:8787/status


In [ ]:
# import os, sys, json

# REPO = "/uscms/home/reshmar/nobackup/IDMe_Run3_Collab/CMSSW_13_0_13/src/iDMe"
# AT   = os.path.join(REPO, "python_analysis/analysisTools")
# TP   = os.path.join(REPO, "python_analysis/configs")

# os.environ["X509_USER_PROXY"] = os.path.expanduser("~/x509up_proxy.pem")
# sys.path.append(AT)

# from lpc_dask import make_lpc_client, ship_analysis_modules, expected_entries
# from analysisTools import Analyzer
# import coffea.util as util

# sample_config = os.path.join(REPO, "python_analysis/configs/sample_configs/bkg_2024_Merged_Full_Run3.json")
# cuts_config   = os.path.join(TP, "cut_configs/cutconfig.py")
# histos_config = os.path.join(TP, "histo_configs/histo_cutflow.py")
# outdir        = os.path.join(REPO, "python_analysis/coffea/SlimPlusMerged2026")
# slimFile = os.path.join(REPO, "python_analysis/configs/sample_configs/bkg_2024_Slim_Run3.json")
# os.makedirs(outdir, exist_ok=True)

# cluster, client = make_lpc_client(n_workers=40)
# print("dashboard:", cluster.dashboard_link)

# try:
#     # On a busy pool the first worker can take 10-15 minutes to appear (measured 822 s).
#     # A long pause here is queueing, not a hang.
#     client.wait_for_workers(1, timeout=1800)

#     ship_analysis_modules(client, configs=[cuts_config, histos_config])

#     analyzer = Analyzer(sample_config, histos_config, cuts_config, model_json=None, max_samples=-1, slimFileList=slimFile,processSlim=True) 
#     out = analyzer.process(execr="dask", dask_client=client)

#     acc, metrics = out                   # savemetrics=True gives a 2-tuple
#     print("entries:", metrics["entries"], "expected:", expected_entries(sample_config))

#     util.save(out, os.path.join(outdir, "Bkg_DASK.coffea"))
# finally:
#     client.close(); cluster.close()      # must run even if the wait times out,
#                                          # or the condor jobs sit holding slots

dashboard: http://localhost:8787/status
shipped mySchema.py
shipped analysisSubroutines.py
shipped corrections.py
shipped analysisTools.py
shipped cuts.py
shipped bkg_histo.py
Running with systematics:  None
[###############################         ] | 78% Completed |  1hr  0min 47.8s

2026-11-06 11:45:48,379 - distributed.scheduler - ERROR - Workers ['tcp://131.225.207.80:10000', 'tcp://131.225.207.80:10004'] did not shut down within 30s; force closing
2026-11-06 11:45:48,388 - distributed.scheduler - ERROR - 2/2 nanny worker(s) did not shut down within 30s: {'tcp://131.225.207.80:10000', 'tcp://131.225.207.80:10004'}
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/distributed/utils.py", line 805, in wrapper
    return await func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/distributed/scheduler.py", line 6624, in restart_workers
    raise TimeoutError(
TimeoutError: 2/2 nanny worker(s) did not shut down within 30s: {'tcp://131.225.207.80:10000', 'tcp://131.225.207.80:10004'}
2026-11-06 11:45:48,390 - distributed.scheduler - ERROR - 2/2 nanny worker(s) did not shut down within 30s: {'tcp://131.225.207.80:10000', 'tcp://131.225.207.80:10004'}
Traceback (most recent call last

[###############################         ] | 78% Completed |  1hr  5min 47.8s

2026-11-06 11:50:48,375 - distributed.scheduler - ERROR - Workers ['tcp://131.225.207.134:10000'] did not shut down within 30s; force closing
2026-11-06 11:50:48,378 - distributed.scheduler - ERROR - 1/1 nanny worker(s) did not shut down within 30s: {'tcp://131.225.207.134:10000'}
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/distributed/utils.py", line 805, in wrapper
    return await func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/distributed/scheduler.py", line 6624, in restart_workers
    raise TimeoutError(
TimeoutError: 1/1 nanny worker(s) did not shut down within 30s: {'tcp://131.225.207.134:10000'}
2026-11-06 11:50:48,379 - distributed.scheduler - ERROR - 1/1 nanny worker(s) did not shut down within 30s: {'tcp://131.225.207.134:10000'}
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/distributed/utils.py", line 805, in wrapper
    return await func

[###############################         ] | 78% Completed |  1hr 10min 33.8s

2026-11-06 11:55:34,318 - distributed.scheduler - ERROR - Status.starting
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/distributed/utils.py", line 805, in wrapper
    return await func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/distributed/scheduler.py", line 6610, in restart_workers
    raise resp
  File "/usr/local/lib/python3.11/site-packages/distributed/utils.py", line 1910, in wait_for
    return await fut
           ^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/distributed/core.py", line 1178, in send_recv_from_rpc
    result = await send_recv(comm=comm, op=key, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/distributed/core.py", line 1043, in send_recv
    raise exc.with_traceback(tb)
  File "/usr/local/lib/python3.11/site-packages/distributed/core.py", line 834, in _handle_comm
    result = await resu

[###############################         ] | 78% Completed |  1hr 30min 56.7s